In [1]:
import timesfm
import chardet
import pandas as pd
import numpy as np

 See https://github.com/google-research/timesfm/blob/master/README.md for updated APIs.
Loaded PyTorch TimesFM, likely because python version is 3.11.13 | packaged by Anaconda, Inc. | (main, Jun  5 2025, 13:03:15) [MSC v.1929 64 bit (AMD64)].


In [2]:
df = pd.read_csv("C:/Users/User/Downloads/sales_data_sample.csv", encoding="cp1252")
df.head(10)

,ORDERNUMBER,QUANTITYORDERED,PRICEEACH,ORDERLINENUMBER,SALES,ORDERDATE,STATUS,QTR_ID,MONTH_ID,YEAR_ID,...,ADDRESSLINE1,ADDRESSLINE2,CITY,STATE,POSTALCODE,COUNTRY,TERRITORY,CONTACTLASTNAME,CONTACTFIRSTNAME,DEALSIZE
0,10100,30,100.00,3,5151.00,1/6/2003 0:00,Shipped,1,1,2003,...,2304 Long Airport Avenue,NaN,Nashua,NH,62005,USA,NaN,Young,Valarie,Medium
1,10100,50,67.80,2,3390.00,1/6/2003 0:00,Shipped,1,1,2003,...,2304 Long Airport Avenue,NaN,Nashua,NH,62005,USA,NaN,Young,Valarie,Medium
2,10100,22,86.51,4,1903.22,1/6/2003 0:00,Shipped,1,1,2003,...,2304 Long Airport Avenue,NaN,Nashua,NH,62005,USA,NaN,Young,Valarie,Small
3,10100,49,34.47,1,1689.03,1/6/2003 0:00,Shipped,1,1,2003,...,2304 Long Airport Avenue,NaN,Nashua,NH,62005,USA,NaN,Young,Valarie,Small
4,10101,25,100.00,4,3782.00,1/9/2003 0:00,Shipped,1,1,2003,...,Lyonerstr. 34,NaN,Frankfurt,NaN,60528,Germany,EMEA,Keitel,Roland,Medium
5,10101,26,100.00,1,3773.38,1/9/2003 0:00,Shipped,1,1,2003,...,Lyonerstr. 34,NaN,Frankfurt,NaN,60528,Germany,EMEA,Keitel,Roland,Medium
6,10101,45,31.20,3,1404.00,1/9/2003 0:00,Shipped,1,1,2003,...,Lyonerstr. 34,NaN,Frankfurt,NaN,60528,Germany,EMEA,Keitel,Roland,Small
7,10101,46,53.76,2,2472.96,1/9/2003 0:00,Shipped,1,1,2003,...,Lyonerstr. 34,NaN,Frankfurt,NaN,60528,Germany,EMEA,Keitel,Roland,Small
8,10102,39,100.00,2,4808.31,1/10/2003 0:00,Shipped,1,1,2003,...,2678 Kingston Rd.,Suite 101,NYC,NY,10022,USA,NaN,Frick,Michael,Medium
9,10102,41,50.14,1,2055.74,1/10/2003 0:00,Shipped,1,1,2003,...,2678 Kingston Rd.,Suite 101,NYC,NY,10022,USA,NaN,Frick,Michael,Small


In [3]:
# Ensure ORDERDATE is datetime
df['ORDERDATE'] = pd.to_datetime(df['ORDERDATE'])
df.head()

,ORDERNUMBER,QUANTITYORDERED,PRICEEACH,ORDERLINENUMBER,SALES,ORDERDATE,STATUS,QTR_ID,MONTH_ID,YEAR_ID,...,ADDRESSLINE1,ADDRESSLINE2,CITY,STATE,POSTALCODE,COUNTRY,TERRITORY,CONTACTLASTNAME,CONTACTFIRSTNAME,DEALSIZE
0,10100,30,100.00,3,5151.00,2003-01-06,Shipped,1,1,2003,...,2304 Long Airport Avenue,NaN,Nashua,NH,62005,USA,NaN,Young,Valarie,Medium
1,10100,50,67.80,2,3390.00,2003-01-06,Shipped,1,1,2003,...,2304 Long Airport Avenue,NaN,Nashua,NH,62005,USA,NaN,Young,Valarie,Medium
2,10100,22,86.51,4,1903.22,2003-01-06,Shipped,1,1,2003,...,2304 Long Airport Avenue,NaN,Nashua,NH,62005,USA,NaN,Young,Valarie,Small
3,10100,49,34.47,1,1689.03,2003-01-06,Shipped,1,1,2003,...,2304 Long Airport Avenue,NaN,Nashua,NH,62005,USA,NaN,Young,Valarie,Small
4,10101,25,100.00,4,3782.00,2003-01-09,Shipped,1,1,2003,...,Lyonerstr. 34,NaN,Frankfurt,NaN,60528,Germany,EMEA,Keitel,Roland,Medium


In [4]:
# --- Step 1: Aggregate per day (start fresh every run) ---
daily_sales = df.groupby('ORDERDATE').agg({
    'SALES': 'sum',
    'QUANTITYORDERED': 'sum',
    'PRICEEACH': 'mean',
    'ORDERLINENUMBER': 'mean'
})

daily_sales.head(10)

,SALES,QUANTITYORDERED,PRICEEACH,ORDERLINENUMBER
ORDERDATE,,,,
2003-01-06,12133.25,151,72.195000,2.5
2003-01-09,11432.34,142,71.240000,2.5
2003-01-10,6864.05,80,75.070000,1.5
2003-01-29,54702.00,541,88.596250,8.5
2003-01-31,44621.96,443,81.683846,7.0
2003-02-11,58871.11,545,85.284000,8.0
2003-02-17,56181.32,675,78.882222,9.5
2003-02-24,25783.76,229,92.801250,4.5
2003-03-03,55245.02,561,82.811250,8.5


In [5]:
# Create a complete date range from min to max date
complete_dates = pd.date_range(start=daily_sales.index.min(), end=daily_sales.index.max(), freq='D')


# Reindex the DataFrame and fill missing values
# Create a new DataFrame with complete dates
complete_sales = daily_sales.reindex(complete_dates)

# Fill missing values in the new DataFrame
complete_sales['SALES'] = complete_sales['SALES'].fillna(0)
complete_sales['QUANTITYORDERED'] = complete_sales['QUANTITYORDERED'].fillna(0)
complete_sales['PRICEEACH'] = complete_sales['PRICEEACH'].fillna(0)
complete_sales['ORDERLINENUMBER'] = complete_sales['ORDERLINENUMBER'].fillna(0)

complete_sales.head(30)

,SALES,QUANTITYORDERED,PRICEEACH,ORDERLINENUMBER
2003-01-06,12133.25,151.0,72.195000,2.5
2003-01-07,0.00,0.0,0.000000,0.0
2003-01-08,0.00,0.0,0.000000,0.0
2003-01-09,11432.34,142.0,71.240000,2.5
2003-01-10,6864.05,80.0,75.070000,1.5
2003-01-11,0.00,0.0,0.000000,0.0
2003-01-12,0.00,0.0,0.000000,0.0
2003-01-13,0.00,0.0,0.000000,0.0
2003-01-14,0.00,0.0,0.000000,0.0
2003-01-15,0.00,0.0,0.000000,0.0


In [6]:
# Reset index to make ORDERDATE a column only if it doesn't exist
if 'ORDERDATE' not in complete_sales.columns:
    complete_sales = complete_sales.reset_index()
    # Only rename if the column is currently named 'index'
    if 'index' in complete_sales.columns:
        complete_sales = complete_sales.rename(columns={'index': 'ORDERDATE'})
    # If the index has a name that's not 'index', use that name
    elif complete_sales.index.name is not None:
        complete_sales = complete_sales.reset_index()
        complete_sales = complete_sales.rename(columns={complete_sales.index.name: 'ORDERDATE'})

complete_sales.head(30)

,ORDERDATE,SALES,QUANTITYORDERED,PRICEEACH,ORDERLINENUMBER
0,2003-01-06,12133.25,151.0,72.195000,2.5
1,2003-01-07,0.00,0.0,0.000000,0.0
2,2003-01-08,0.00,0.0,0.000000,0.0
3,2003-01-09,11432.34,142.0,71.240000,2.5
4,2003-01-10,6864.05,80.0,75.070000,1.5
5,2003-01-11,0.00,0.0,0.000000,0.0
6,2003-01-12,0.00,0.0,0.000000,0.0
7,2003-01-13,0.00,0.0,0.000000,0.0
8,2003-01-14,0.00,0.0,0.000000,0.0
9,2003-01-15,0.00,0.0,0.000000,0.0


In [7]:
# Add dynamic categorical covariate: weekday
# Monday=0, Sunday=6
complete_sales["WEEKDAY"] = pd.to_datetime(complete_sales["ORDERDATE"]).dt.dayofweek

complete_sales.head(10)

,ORDERDATE,SALES,QUANTITYORDERED,PRICEEACH,ORDERLINENUMBER,WEEKDAY
0,2003-01-06,12133.25,151.0,72.195,2.5,0
1,2003-01-07,0.00,0.0,0.000,0.0,1
2,2003-01-08,0.00,0.0,0.000,0.0,2
3,2003-01-09,11432.34,142.0,71.240,2.5,3
4,2003-01-10,6864.05,80.0,75.070,1.5,4
5,2003-01-11,0.00,0.0,0.000,0.0,5
6,2003-01-12,0.00,0.0,0.000,0.0,6
7,2003-01-13,0.00,0.0,0.000,0.0,0
8,2003-01-14,0.00,0.0,0.000,0.0,1
9,2003-01-15,0.00,0.0,0.000,0.0,2


In [8]:
split_idx = int(len(complete_sales)*0.94)
train_df = complete_sales[:split_idx]
test_df = complete_sales[split_idx:]
print(train_df.shape, test_df.shape)

(824, 6) (53, 6)


In [9]:
import os
# Verify files exist
config_path = "C:/Users/User/OneDrive/Documents/GitHub/FutureReady_Invoicing/services/ai/timesfm_export/timesfm_config.json"
model_weights_path = "C:/Users/User/OneDrive/Documents/GitHub/FutureReady_Invoicing/services/ai/timesfm_export/timesfm_model.pt"
CHECKPOINT_DIR = "C:/Users/User/OneDrive/Documents/GitHub/FutureReady_Invoicing/services/ai/times_checkpoint"
if not os.path.exists(config_path):
    raise FileNotFoundError(f"Config file {config_path} not found")
if not os.path.exists(model_weights_path):
    raise FileNotFoundError(f"Model weights {model_weights_path} not found")
checkpoint_file = os.path.join(CHECKPOINT_DIR, "torch_model.ckpt")
if not os.path.exists(checkpoint_file):
    raise FileNotFoundError(f"Checkpoint file {checkpoint_file} not found")

In [10]:
import json
hparams_dict = None
try:
    with open(config_path, "r", encoding="utf-8") as f:
        hparams_dict = json.load(f)
    print(f"Loaded config from {config_path}")
except UnicodeDecodeError:
    print(f"UnicodeDecodeError: Trying to read {config_path} with cp1252 encoding")
    try:
        with open(config_path, "r", encoding="cp1252") as f:
            hparams_dict = json.load(f)
        print(f"Loaded config with cp1252 encoding")
    except Exception as e:
        print(f"Failed to read {config_path}: {e}")
        print("Regenerating default hparams")
        hparams_dict = {
            "backend": "cpu",
            "per_core_batch_size": 1,
            "horizon_len": 53,
            "num_layers": 50,
            "use_positional_embedding": False,
            "context_len": 512,
        }
        # Save corrected config
        with open(config_path, "w", encoding="utf-8") as f:
            json.dump(hparams_dict, f, indent=4)
        print(f"Regenerated and saved config to {config_path}")
except Exception as e:
    print(f"Error loading config {config_path}: {e}")
    raise

Loaded config from C:/Users/User/OneDrive/Documents/GitHub/FutureReady_Invoicing/services/ai/timesfm_export/timesfm_config.json


In [11]:
import json
# Verify read permissions
try:
    with open(config_path, "r") as f:
        hparams_dict = json.load(f)
    with open(checkpoint_file, "rb") as f:
        pass  # Test read access
    with open(model_weights_path, "rb") as f:
        pass  # Test read access
    print(f"Read permissions verified for {CHECKPOINT_DIR}, {model_weights_path}, and {config_path}")
except PermissionError as e:
    print(f"PermissionError: Cannot read files: {e}")
    print("Try running as administrator or check file permissions with: icacls \"path\"")
    raise

Read permissions verified for C:/Users/User/OneDrive/Documents/GitHub/FutureReady_Invoicing/services/ai/times_checkpoint, C:/Users/User/OneDrive/Documents/GitHub/FutureReady_Invoicing/services/ai/timesfm_export/timesfm_model.pt, and C:/Users/User/OneDrive/Documents/GitHub/FutureReady_Invoicing/services/ai/timesfm_export/timesfm_config.json


In [12]:
import torch
try:
    state_dict = torch.load("C:/Users/User/OneDrive/Documents/GitHub/FutureReady_Invoicing/services/ai/timesfm_export/timesfm_model.pt", weights_only=False)
    print("Keys in state_dict:", list(state_dict.keys())[:5])  # Print first 5 keys
except Exception as e:
    print(f"Error inspecting timesfm_model.pt: {e}")

Keys in state_dict: ['input_ff_layer.hidden_layer.0.weight', 'input_ff_layer.hidden_layer.0.bias', 'input_ff_layer.output_layer.weight', 'input_ff_layer.output_layer.bias', 'input_ff_layer.residual_layer.weight']


In [13]:
import pickle
# --- Step 3: Reload Model ---
try:
    tfm_reloaded = timesfm.TimesFm(
        hparams=timesfm.TimesFmHparams(**hparams_dict),
        checkpoint=timesfm.TimesFmCheckpoint(path=CHECKPOINT_DIR + "/torch_model.ckpt"),
    )
    # Try loading with weights_only=True
    try:
        tfm_reloaded._model.load_state_dict(torch.load(model_weights_path, weights_only=True))
        print("Reloaded model successfully with weights_only=True")
    except pickle.UnpicklingError as e:
        print(f"UnpicklingError with weights_only=True: {e}")
        print("Loading with weights_only=False (safe since timesfm_model.pt is from a trusted source)")
        tfm_reloaded._model.load_state_dict(torch.load(model_weights_path, weights_only=False))
        print("Reloaded model successfully with weights_only=False")
    tfm_reloaded._model.eval()
except PermissionError as e:
    print(f"PermissionError reloading checkpoint: {e}")
    print(f"Ensure {checkpoint_file} is readable")
    raise
except Exception as e:
    print(f"Error reloading model: {e}")
    print("Consider regenerating timesfm_model.pt with weights_only compatible format")
    raise

Reloaded model successfully with weights_only=True


In [ ]:
context_len = 512  
horizon_len = 53
batch_size = 1   

In [ ]:
from collections import defaultdict
from datetime import timedelta

def get_batched_data_fn(
    batch_size: int = 1,
    context_len: int = 512,
    horizon_len: int = 53,
):
    examples = defaultdict(list)

    num_examples = 0
    sub_df = complete_sales.copy()
    sub_df['ds'] = complete_sales['ORDERDATE']  # Full df for context from train

    # Calculate test_start_idx
    test_start_idx = len(complete_sales) - len(test_df)  # 824

    # Start range to ensure only the forecast covering the exact test set
    start_begin = max(0, test_start_idx - context_len)
    for start in range(start_begin, len(sub_df) - (context_len + horizon_len) + 1, horizon_len):
        num_examples += 1
        context_end = start + context_len
        examples["inputs"].append(sub_df["SALES"][start:context_end].tolist())
        examples["quantity_ordered"].append(sub_df["QUANTITYORDERED"][start:context_end + horizon_len].tolist())
        examples["price_each"].append(sub_df["PRICEEACH"][start:context_end + horizon_len].tolist())
        examples["order_line_number"].append(sub_df["ORDERLINENUMBER"][start:context_end + horizon_len].tolist())
        examples["week_day"].append(sub_df["WEEKDAY"][start:context_end + horizon_len].tolist())
        examples["outputs"].append(sub_df["SALES"][context_end:context_end + horizon_len].tolist())

    def data_fn():
        for i in range(1 + (num_examples - 1) // batch_size):
            yield {k: v[(i * batch_size):((i + 1) * batch_size)] for k, v in examples.items()}

    return data_fn

In [16]:
input_data = get_batched_data_fn(batch_size=batch_size, context_len=context_len, horizon_len=horizon_len)
print(f"Type of input_data: {type(input_data)}")
if not callable(input_data):
    raise TypeError("input_data is not callable. Ensure get_batched_data_fn returns a generator function")
metrics = defaultdict(list)

Type of input_data: <class 'function'>


In [17]:
import time
import jax
for i, example in enumerate(input_data()):
    # Raw forecast
    raw_forecast, _ = tfm_reloaded.forecast(
        inputs=example["inputs"],
        freq=[0] * len(example["inputs"])
    )
    start_time = time.time()
    # Forecast with covariates
    cov_forecast, ols_forecast = tfm_reloaded.forecast_with_covariates(
        inputs=example["inputs"],
        dynamic_numerical_covariates={
            "quantity_ordered": example["quantity_ordered"],
            "price_each": example["price_each"],
            "order_line_number": example["order_line_number"],
        },
        dynamic_categorical_covariates={
            "week_day": example["week_day"],
        },
        static_numerical_covariates={},
        static_categorical_covariates={},
        freq=[0] * len(example["inputs"]),
        xreg_mode="xreg + timesfm",
        ridge=0.0,
        force_on_cpu=False,
        normalize_xreg_target_per_input=True,
    )
    print(f"\rFinished batch {i} in {time.time() - start_time} seconds", end="")

In [19]:
cov_forecast[0]

NameError: name 'cov_forecast' is not defined

In [18]:
import matplotlib.pyplot as plt
# Visualization
print(f"Forecast for test set (length {len(cov_forecast[0])}): {cov_forecast[0]}")

days_before = 10 # Number of days before the split to include  
days_after = 53 # Number of days after the split (covers full test set)

# Train/test split line (using the first date of test_df)
split_date = test_df['ORDERDATE'].iloc[0]
# Calculate window boundaries
window_start = split_date - timedelta(days=days_before)
window_end = split_date + timedelta(days=days_after)

# Filter historical data to the window
historical_window = complete_sales[(complete_sales['ORDERDATE'] >= window_start) & (complete_sales['ORDERDATE'] <= window_end)]

# Create a DataFrame for the forecast
forecast_dates = test_df['ORDERDATE'].values  # Dates for the test set (53 days)
forecast_values = cov_forecast[0]  # Forecasted sales (length 53)
forecast_df = pd.DataFrame({'ds': forecast_dates, 'SALES': forecast_values})
# Filter forecast to the window (should include all forecast data since days_after=53)
forecast_window = forecast_df[(forecast_df['ds'] >= window_start) & (forecast_df['ds'] <= window_end)]

# Plot the entire dataset and forecast
plt.figure(figsize=(12, 6))
# Historical data (complete_sales)
plt.plot(historical_window['ORDERDATE'], historical_window['SALES'], label='Historical Sales', color='blue')
# Forecast
plt.plot(forecast_window['ds'], forecast_window['SALES'], label='Forecasted Sales', color='red', linestyle='--')

# Train/test split line (only if within window)
if window_start <= split_date <= window_end:
    plt.axvline(x=split_date, color='green', linestyle=':', label='Train/Test Split')
plt.xlabel('Date')
plt.ylabel('Sales')
plt.title(f'Historical and Forecasted Sales (Window: {days_before} days before to {days_after} days)')
plt.legend()
plt.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

NameError: name 'cov_forecast' is not defined